# Submission 5 — Project Deployment (5%)

**Course:** RBB2013 / FFM2063 / FEM2063 — Digital Twin, May 2026
**Project:** SmartClean Twin — a software-emulated Digital Twin of a mobile
inspection and cleaning robot (project topic 2)
**Repository:** https://github.com/KAI-UTP/smartclean-twin
**Presentation & demo video:** [https://youtu.be/zEq7L-ivMLA](https://youtu.be/zEq7L-ivMLA)

**Team Members**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |

> **How to reproduce:** start the stack with `docker compose up -d` (8 containers),
> then run this notebook top to bottom. All outputs below were produced against
> the running system.


## 1. Executive Summary

The SmartClean Twin is deployed as **eight containers** — five custom
microservices plus a message broker, a time-series database and a dashboard
server — orchestrated by a single `docker compose up -d`.

| Aspect | Result |
|---|---|
| Microservice decomposition | 5 custom services, each with one responsibility (Section 3) |
| Interface contract | All 16 communicating pairs specified with route, port, protocol, data format and initiation/conclusion conditions (Sections 4–5) |
| Containerization | One Dockerfile per service; images build reproducibly, AI models trained at build time (Section 6) |
| Deployment | All 8 containers healthy from one command; verified live (Section 8) |
| Scaling | `telemetry-ingestion` scaled to 2 concurrent replicas; verified live (Section 9) |
| Persistence | Data survives an InfluxDB container restart; 3/3 automated tests pass (Section 10) |
| Full-flow test suite | 11 system tests covering the complete twin flow, all passing (Section 11) |
| Fault tolerance | Services reconnect automatically; verified in Submission 4 Section 10 |


## 2. Decomposition Rationale

The system is partitioned so that each service owns exactly one concern and can
fail, restart, be replaced or be scaled independently. Two rules guided the
split:

1. **One responsibility per service.** Validation is separate from state
   derivation, which is separate from prediction, so a change to a state rule
   cannot break data ingestion.
2. **Communicate through the broker, not directly.** No custom service calls
   another custom service synchronously. Every inter-service data flow goes
   through MQTT topics, which decouples lifecycles: a service can be stopped and
   restarted without its peers noticing anything except a gap in messages.

The practical benefit was demonstrated during development: the AI service was
added in Sprint 2 by subscribing to an existing topic, with **zero changes** to
any Sprint 1 service. The same property means the simulator could be replaced by
a driver for a real robot without touching the other seven containers.


## 3. Microservice Specification — Function and Operation

| Service | Container | Port | Function | Operation |
|---|---|---|---|---|
| robot-simulator | `smartclean-simulator` | 8004 | Emulates the physical asset | 1 s physics tick: lawnmower path, battery discharge and dock charging, obstacle scan; publishes telemetry; subscribes to command topics and returns an ACK; exposes a fault-injection REST API |
| mosquitto | `smartclean-mosquitto` | 1883 | Message broker | Routes all telemetry, state, prediction, alert, command and ACK traffic between services; persistence enabled |
| telemetry-ingestion | scalable (host range 8001–8011) | 8001 | Validation and storage gateway | Subscribes to raw telemetry, validates against the shared schema, rejects and counts invalid messages, writes valid ones to InfluxDB, republishes on the validated topic |
| state-engine | `smartclean-state-engine` | 8002 | Digital Twin state derivation | Consumes validated telemetry, applies rules to produce 11 state variables and alarms, publishes state and alerts, writes state to InfluxDB |
| ai-service | `smartclean-ai-service` | 8003 | Intelligence layer | Loads five models at start-up, scores every validated telemetry message, maintains rolling windows for forecasts, publishes predictions, writes them to InfluxDB, serves `POST /whatif` |
| command-api | `smartclean-command-api` | 8000 | Operator control interface | Accepts `POST /api/v1/commands`, validates the robot ID and command, publishes to the command topic, waits for the ACK, returns the result; keeps a command history |
| influxdb | `smartclean-influxdb` | 8086 | Time-series store | Persists four measurements on the `influxdb_data` named volume; serves Flux queries |
| grafana | `smartclean-grafana` | 3001 → 3000 | Visualization server | Serves the provisioned 28-panel dashboard, querying InfluxDB every 5 s |

Every custom service additionally exposes `GET /health` returning status, uptime
and service-specific counters, which is what makes the liveness check in
Section 8 possible.


## 4. Interface Contract

`docs/api-contract.md` specifies **all 16 communicating pairs**. For each pair it
records the route or topic, the port, the protocol, the data format, and the
conditions under which the communication is **initiated** and **concluded**.

The 16 pairs are:

| # | Pair | Transport |
|---|---|---|
| 1 | robot-simulator → mosquitto (raw telemetry) | MQTT 1883 |
| 2 | mosquitto → telemetry-ingestion (raw telemetry) | MQTT 1883 |
| 3 | telemetry-ingestion → mosquitto (validated telemetry) | MQTT 1883 |
| 4 | telemetry-ingestion → influxdb (write) | HTTP 8086 |
| 5 | mosquitto → state-engine (validated telemetry) | MQTT 1883 |
| 6 | state-engine → mosquitto (state + alerts) | MQTT 1883 |
| 7 | state-engine → influxdb (write) | HTTP 8086 |
| 8 | mosquitto → ai-service (validated telemetry) | MQTT 1883 |
| 9 | ai-service → mosquitto (predictions) | MQTT 1883 |
| 10 | ai-service → influxdb (write) | HTTP 8086 |
| 11 | operator → command-api (command injection) | HTTP 8000 |
| 12 | command-api → mosquitto → simulator (command + ACK) | MQTT 1883 |
| 13 | operator → robot-simulator (fault injection) | HTTP 8004 |
| 14 | grafana → influxdb (Flux queries) | HTTP 8086 |
| 15 | operator → all services (health checks) | HTTP 8000–8004 |
| 16 | omniverse → influxdb (3D visualization) | HTTP 8086 |

## 5. Contract Detail — Representative Pairs

**Pair 1 — robot-simulator → mosquitto**

| Field | Value |
|---|---|
| Route / topic | `smartclean/SCR01/telemetry/raw` |
| Port / protocol | 1883 / MQTT, QoS 1 |
| Data format | JSON, `TelemetryMessage` schema, 16 sensor and pose fields |
| Initiated | On simulator start-up, after the MQTT connect succeeds |
| Concluded | On SIGTERM / SIGINT received by the simulator |

**Pair 4 — telemetry-ingestion → influxdb**

| Field | Value |
|---|---|
| Route | `POST /api/v2/write?org=smartclean&bucket=smartclean_twin` |
| Port / protocol | 8086 / HTTP with Influx line protocol |
| Data format | Line protocol, measurement `robot_telemetry`, tag `robot_id` |
| Initiated | Per valid telemetry message, synchronous write |
| Concluded | On write acknowledgement, HTTP 204 |

**Pair 12 — command-api → mosquitto → robot-simulator**

| Field | Value |
|---|---|
| Route / topic | `smartclean/SCR01/command/motion`; reply on `smartclean/SCR01/ack` |
| Port / protocol | 1883 / MQTT, QoS 1 |
| Data format | JSON `CommandMessage` outbound, JSON `AckMessage` inbound |
| Initiated | When the Command API receives an HTTP POST from the operator |
| Concluded | When the matching ACK arrives from the simulator, or on timeout |

The remaining 13 pairs are documented in the same form in
`docs/api-contract.md`, together with a port summary table.


## 6. Containerization Strategy

Each custom service has its own Dockerfile following the same pattern: a slim
Python 3.11 base, the shared library copied in, dependencies installed from a
pinned `requirements.txt`, then the service code. Dependencies are pinned to
exact versions so that an image built today and an image built next month are
functionally identical.

Two container-level decisions are worth highlighting:

**AI models are trained during the image build.** The `ai-service` Dockerfile
runs `train_model.py` as a build step, so the model artefacts are baked into the
image alongside the code that produced them. They cannot drift apart, and the
image is reproducible from source alone.

**Grafana's dashboard and datasource are baked in, not bind-mounted.** The
Grafana image is built from `grafana/Dockerfile`, copying the provisioning
directory and the dashboard JSON into the image. This makes the dashboard
version-controlled and reproducible, and it also avoided a Docker Desktop
limitation with bind mounts on paths containing spaces.

Configuration is supplied by environment variables in `docker-compose.yml`
(broker host, InfluxDB URL, token, organisation, bucket, robot ID, telemetry
interval), so no service contains a hard-coded environment-specific value.


## 7. Orchestration

`docker-compose.yml` defines the eight services, one network, and two named
volumes (`influxdb_data`, `grafana_data`, plus `mosquitto_data` for broker
persistence). Two orchestration features are used deliberately:

- **Health checks and ordered start-up.** Mosquitto and InfluxDB declare
  `healthcheck` commands, and dependent services declare
  `depends_on: condition: service_healthy`. Services therefore do not start
  until their dependencies can actually serve traffic, which eliminates the
  race conditions that make a naive compose stack fail on a cold start.
- **`restart: unless-stopped`.** A crashed service is restarted automatically,
  so a transient failure does not require manual intervention.

The `telemetry-ingestion` service intentionally declares **no fixed container
name and a host port range** (`8001-8011:8001`), because a container name and a
single host port are both unique resources and would prevent a second replica
from starting. This is what makes the scaling demonstration in Section 9
possible.


## 8. Live Evidence — Deployment

All eight containers are started by one command. The cells below show the
running containers and then confirm that every custom service is actually
serving traffic, not merely running.

In [1]:
import subprocess

r = subprocess.run(["docker", "compose", "ps", "--format", "{{.Name}}\t{{.Status}}"],
                   capture_output=True, text=True, cwd=".")
print("Container status:")
for line in r.stdout.splitlines():
    if line.strip():
        print("  " + line)


Container status:
  smartclean-ai-service	Up 44 minutes
  smartclean-command-api	Up 44 minutes
  smartclean-grafana	Up 44 minutes
  smartclean-influxdb	Up About a minute (healthy)
  smartclean-mosquitto	Up 37 seconds (healthy)
  smartclean-simulator	Up 45 minutes
  smartclean-state-engine	Up 44 minutes
  smartclean-twin-telemetry-ingestion-1	Up 44 minutes


In [2]:
import json, urllib.request

SERVICES = {
    "command-api": 8000,
    "telemetry-ingestion": 8001,
    "state-engine": 8002,
    "ai-service": 8003,
    "robot-simulator": 8004,
}
print("Service liveness (HTTP /health):")
for name, port in SERVICES.items():
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/health", timeout=5) as resp:
            h = json.loads(resp.read())
            extra = {k: v for k, v in h.items() if k not in ("status", "timestamp")}
            print(f"  {name:22s} HTTP {resp.status}  {h.get('status','?'):9s} {extra}")
    except Exception as exc:
        print(f"  {name:22s} UNREACHABLE ({exc})")


Service liveness (HTTP /health):
  command-api            HTTP 200  healthy   {'uptime_s': 2697.5, 'commands_issued': 36}
  telemetry-ingestion    HTTP 200  healthy   {'uptime_s': 2697.5, 'received': 2686, 'valid': 2686, 'invalid': 0, 'valid_rate_pct': 100.0}
  state-engine           HTTP 200  healthy   {'uptime_s': 2697.5, 'messages_processed': 2694, 'cleaning_coverage_pct': 50.85, 'last_message': '2026-07-26T23:54:05.118235+00:00'}
  ai-service             HTTP 200  healthy   {'uptime_s': 2697.2, 'model_loaded': True, 'predictions_made': 2692}
  robot-simulator        HTTP 200  healthy   {'robot_id': 'SCR01', 'battery_soc': 99.9, 'mode': 'IDLE', 'paused': False, 'stopped': True}


## 9. Live Evidence — Scaling

**Which services can scale, and why.** `telemetry-ingestion` is horizontally
scalable because it is **stateless** — it holds no data between messages — and
because MQTT delivers each published message to **every** subscribed client, so
additional replicas simply add parallel validation and write capacity.

The other services are deliberately kept single-instance: the state engine and
the AI service maintain in-memory state (previous twin state, rolling windows)
that would diverge across replicas, and the simulator represents one physical
asset, so a second copy would be meaningless.

The cell below scales ingestion to two replicas, shows both running, then scales
back to one.

In [3]:
import subprocess, time

print("Scaling telemetry-ingestion to 2 replicas ...")
subprocess.run(["docker", "compose", "up", "--scale", "telemetry-ingestion=2", "-d"],
               capture_output=True, text=True, cwd=".")
time.sleep(15)

r = subprocess.run(["docker", "compose", "ps", "--format", "{{.Name}}\t{{.Status}}"],
                   capture_output=True, text=True, cwd=".")
print("\nIngestion replicas now running:")
for line in r.stdout.splitlines():
    if "telemetry" in line:
        print("  " + line)

print("\nScaling back to 1 replica ...")
subprocess.run(["docker", "compose", "up", "--scale", "telemetry-ingestion=1", "-d"],
               capture_output=True, text=True, cwd=".")
print("Done.")


Scaling telemetry-ingestion to 2 replicas ...



Ingestion replicas now running:
  smartclean-twin-telemetry-ingestion-1	Up 45 minutes
  smartclean-twin-telemetry-ingestion-2	Up 15 seconds

Scaling back to 1 replica ...


Done.


## 10. Live Evidence — Persistence of Data and State

**Mechanism.** InfluxDB writes to the named Docker volume `influxdb_data`, whose
lifetime is independent of the container's. Destroying and recreating the
container therefore does not destroy the data. InfluxDB additionally provides
durability guarantees that a plain JSON file cannot: a partially written file
after a crash is a corrupt record, whereas the database recovers to a consistent
state.

**Test method.** For each of the three measurements the test counts the rows in
a **frozen** time window, restarts the InfluxDB container, waits for it to become
healthy again, and re-counts over the *same* window. The window must be frozen
rather than relative: a sliding `-60m` range would drop the oldest seconds while
the restart runs and would look like data loss when none occurred.

Pass condition: the count after the restart is greater than or equal to the
count before.

In [4]:
import subprocess, sys

r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/system/test_persistence.py", "-v", "--no-header", "-s"],
                   capture_output=True, text=True, cwd=".")
print(r.stdout[-2200:])
print("Exit code:", r.returncode, " (0 = persistence proven for all three measurements)")


============================= test session starts =============================
collecting ... collected 3 items

tests/system/test_persistence.py::test_data_persists_after_influxdb_restart 
[BEFORE RESTART] robot_telemetry row count: 43200
[RESTART] InfluxDB container restarted
[RECOVERY] InfluxDB healthy after 1s
[AFTER RESTART] robot_telemetry row count: 43200
[PASS] Persistence confirmed: 43200 rows preserved across restart (>= 43200 before restart)
PASSED
tests/system/test_persistence.py::test_state_data_persists_after_influxdb_restart smartclean-influxdb

[BEFORE] robot_state count: 27050
[AFTER] robot_state count: 27050
[PASS] robot_state persisted: 27050 rows
PASSED
tests/system/test_persistence.py::test_prediction_data_persists_after_influxdb_restart smartclean-influxdb

[BEFORE] robot_prediction count: 27334
[AFTER] robot_prediction count: 27334
[PASS] robot_prediction persisted: 27334 rows
PASSED

============================= 3 passed in 12.76s =============================

## 11. Live Evidence — Complete Digital Twin Flow Test Suite

The system suite exercises the whole twin rather than any single service:
service health, telemetry reaching storage, twin state derivation, the command
path with acknowledgement, and the three fault scenarios (obstacle, motor
overload, low battery). Passing this suite is the evidence that the deployed
system works as an integrated Digital Twin, not merely as eight running
processes.

In [5]:
import os, subprocess, sys

env = dict(os.environ, INTEGRATION_TEST="1")
r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/system/test_full_flow.py", "-v", "--no-header"],
                   capture_output=True, text=True, cwd=".", env=env)
print(r.stdout[-2400:])
print("Exit code:", r.returncode)


============================= test session starts =============================
collecting ... collected 11 items

tests/system/test_full_flow.py::TestServiceHealth::test_command_api_health PASSED [  9%]
tests/system/test_full_flow.py::TestServiceHealth::test_ingestion_health PASSED [ 18%]
tests/system/test_full_flow.py::TestServiceHealth::test_state_engine_health PASSED [ 27%]
tests/system/test_full_flow.py::TestServiceHealth::test_ai_service_health PASSED [ 36%]
tests/system/test_full_flow.py::TestServiceHealth::test_simulator_health PASSED [ 45%]
tests/system/test_full_flow.py::TestCommandFlow::test_pause_command_returns_ack PASSED [ 54%]
tests/system/test_full_flow.py::TestCommandFlow::test_resume_after_pause PASSED [ 63%]
tests/system/test_full_flow.py::TestCommandFlow::test_command_history_grows PASSED [ 72%]
tests/system/test_full_flow.py::TestObstacleEmergencyScenario::test_inject_obstacle_and_check_state PASSED [ 81%]
tests/system/test_full_flow.py::TestMotorOverloadScenario::

## 12. Fault Tolerance and Recovery

Three mechanisms provide resilience, and each has been exercised:

| Mechanism | Behaviour | Evidence |
|---|---|---|
| MQTT reconnection with exponential back-off | A service whose broker disappears retries with increasing delay, capped at 30 s, and resumes when the broker returns | Submission 4, Section 10: broker stopped, tests fail, broker restarted, tests pass again with no code change or manual restart |
| `restart: unless-stopped` | A crashed container is restarted automatically by Docker | Observed during development after a host restart |
| Rule-based AI fallback | If model artefacts cannot be loaded, the AI service serves threshold-based predictions and reports `model_used = "rule_fallback"` | Code path in `predictor.py`; the twin stays observable during a degraded state rather than going dark |

Because storage is on a named volume, none of these recovery paths risks data
loss for records already written.


## 13. Discussion and Limitations

**What the deployment achieves.** One command brings up an eight-container
system in dependency order, with health-gated start-up, automatic restart,
durable storage and a horizontally scalable ingestion tier. Every interface
between every pair of components is documented, including when each
communication begins and ends. The complete twin flow, the persistence
guarantee and the scaling capability are all verified by automated tests rather
than asserted.

**Limitations.**

1. **Single host, single broker.** `docker compose` orchestrates one machine.
   Mosquitto and InfluxDB are single instances and therefore single points of
   failure; a production deployment would use a clustered broker and a
   replicated database, or Kubernetes with multiple nodes.
2. **No authentication or TLS.** MQTT allows anonymous connections and the REST
   APIs are unauthenticated. This is the most significant gap relative to a real
   deployment and would be the first thing addressed.
3. **Grafana uses default credentials** (`admin`/`admin`).
4. **Only ingestion scales.** Making the state engine or AI service scalable
   would require externalising their in-memory state, for example into Redis.
5. **CD stops at image build.** Images are built and validated in CI but not
   published to a registry or deployed automatically to an environment.
6. **No resource limits.** Containers declare no CPU or memory limits, so one
   misbehaving service could starve the others on a shared host.


## 14. Conclusion

The Digital Twin is deployed as eight individually containerised components with
one clearly defined responsibility each, brought up by a single
`docker compose up -d` with health-gated ordering and automatic restart. The
interface contract covers all 16 communicating pairs with route, port, protocol,
data format and initiation/conclusion conditions. This notebook demonstrates
live: all eight containers healthy and serving `/health`; the ingestion service
scaled to two concurrent replicas and back; persistence of telemetry, state and
prediction data across an InfluxDB container restart (3/3 tests); and the
complete Digital Twin flow test suite passing (11/11 tests).
